# RecKAN vs. Baseline KANs — Core Framework (models + argparse main)

In [ ]:

"""
RecKAN vs. 3 baseline KANs — Improved version with better architecture
Based on the lightweight implementation that achieved 96.82% on MNIST
"""
import os
import csv
import time
import math
import argparse

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

class RecursiveBasis(nn.Module):
    
    def __init__(self, K, B=3.0, init=(0.0, 0.5, 0.0, 0.0, -0.5)):
        super().__init__()
        self.K = K
        self.B = B
        
        atanh = lambda v: 0.5 * math.log((1 + v / B) / (1 - v / B)) if abs(v) < B else 0.0
        self.a_raw = nn.Parameter(torch.tensor(atanh(init[0])))
        self.b_raw = nn.Parameter(torch.tensor(atanh(init[1])))
        self.c_raw = nn.Parameter(torch.tensor(atanh(init[2])))
        self.d_raw = nn.Parameter(torch.tensor(atanh(init[3])))
        self.e_raw = nn.Parameter(torch.tensor(atanh(init[4])))

    def forward(self, x):
        a = self.B * torch.tanh(self.a_raw)
        b = self.B * torch.tanh(self.b_raw)
        c = self.B * torch.tanh(self.c_raw)
        d = self.B * torch.tanh(self.d_raw)
        e = self.B * torch.tanh(self.e_raw)
        
        R0 = torch.zeros_like(x)
        R1 = torch.ones_like(x)
        basis = [R0, R1]
        
        for n in range(1, self.K):
            alpha = a * x ** 2 + b * x + c
            beta = d * x + e
            Rn1 = alpha * basis[-1] + beta * basis[-2]
            
            basis.append(Rn1)
            
        return torch.stack(basis, dim=-1) 


class RecKANLayer(nn.Module):
    def __init__(self, d_in, d_out, K, basis: RecursiveBasis):
        super().__init__()
        self.d_in, self.d_out, self.K = d_in, d_out, K
        self.basis = basis
        self.W = nn.Parameter(torch.randn(d_in, d_out, K + 1) * (1.0 / math.sqrt(d_in * (K + 1))))

    def forward(self, x):
        x_clamped = x.clamp(-2.0, 2.0)
        R = self.basis(x_clamped)               # (B, d_in, K+1)
        y = torch.einsum('bik,iok->bo', R, self.W)
        return y


class RecKAN(nn.Module):
    def __init__(self, dims, K, use_norm=True, dropout_rate=0.1):
        super().__init__()
        self.basis = RecursiveBasis(K)
        self.layers = nn.ModuleList()
        self.norms = nn.ModuleList()
        self.drops = nn.ModuleList()
        self.use_norm = use_norm
        
        # Create layers
        for i in range(len(dims) - 1):
            self.layers.append(RecKANLayer(dims[i], dims[i + 1], K, self.basis))
            # Add LayerNorm and Dropout after each layer except the last
            if i < len(dims) - 2:
                if use_norm:
                    self.norms.append(nn.LayerNorm(dims[i + 1]))
                self.drops.append(nn.Dropout(dropout_rate))
            else:
                if use_norm:
                    self.norms.append(nn.Identity())
                self.drops.append(nn.Identity())

    def forward(self, x):
        for i, layer in enumerate(self.layers):
            x = layer(x)
            if i < len(self.layers) - 1:
                # After hidden layers: norm + dropout + tanh
                x = self.norms[i](x)
                x = self.drops[i](x)
                x = torch.tanh(x)
        return x


class ChebyKANLayer(nn.Module):
    def __init__(self, d_in, d_out, K):
        super().__init__()
        self.d_in, self.d_out, self.K = d_in, d_out, K
        self.W = nn.Parameter(torch.randn(d_in, d_out, K + 1) * (1.0 / math.sqrt(d_in * (K + 1))))

    def forward(self, x):
        h = x.clamp(-2.0, 2.0)
        T0 = torch.ones_like(h)
        T1 = h
        basis = [T0, T1]
        for n in range(1, self.K):
            Tn1 = 2 * h * basis[-1] - basis[-2]
            basis.append(Tn1)
        R = torch.stack(basis, dim=-1)
        y = torch.einsum('bik,iok->bo', R, self.W)
        return y


class ChebyKAN(nn.Module):
    def __init__(self, dims, K, use_norm=True, dropout_rate=0.1):
        super().__init__()
        self.layers = nn.ModuleList()
        self.norms = nn.ModuleList()
        self.drops = nn.ModuleList()
        self.use_norm = use_norm
        
        for i in range(len(dims) - 1):
            self.layers.append(ChebyKANLayer(dims[i], dims[i + 1], K))
            if i < len(dims) - 2:
                if use_norm:
                    self.norms.append(nn.LayerNorm(dims[i + 1]))
                self.drops.append(nn.Dropout(dropout_rate))
            else:
                if use_norm:
                    self.norms.append(nn.Identity())
                self.drops.append(nn.Identity())

    def forward(self, x):
        for i, layer in enumerate(self.layers):
            x = layer(x)
            if i < len(self.layers) - 1:
                x = self.norms[i](x)
                x = self.drops[i](x)
                x = torch.tanh(x)
        return x


class JacobiShape(nn.Module):
    def __init__(self, init_alpha=0.0, init_beta=0.0, B=2.0):
        super().__init__()
        self.B = B
        atanh = lambda v: 0.5 * math.log((1 + v / B) / (1 - v / B)) if abs(v) < B else 0.0
        self.alpha_raw = nn.Parameter(torch.tensor(atanh(init_alpha)))
        self.beta_raw = nn.Parameter(torch.tensor(atanh(init_beta)))

    def get(self):
        return self.B * torch.tanh(self.alpha_raw), self.B * torch.tanh(self.beta_raw)


class JacobiKANLayer(nn.Module):
    def __init__(self, d_in, d_out, K, shape: JacobiShape):
        super().__init__()
        self.d_in, self.d_out, self.K = d_in, d_out, K
        self.shape = shape
        self.W = nn.Parameter(torch.randn(d_in, d_out, K + 1) * (1.0 / math.sqrt(d_in * (K + 1))))

    def forward(self, x):
        h = x.clamp(-2.0, 2.0)  # Use clamp instead of tanh
        al, be = self.shape.get()
        P0 = torch.ones_like(h)
        basis = [P0]
        if self.K >= 1:
            P1 = 0.5 * (al - be) + 0.5 * (al + be + 2) * h
            basis.append(P1)
        for n in range(1, self.K):
            n_ = float(n)
            a1 = 2 * (n_ + 1) * (n_ + al + be + 1) * (2 * n_ + al + be)
            a2 = (2 * n_ + al + be + 1) * (al ** 2 - be ** 2)
            a3 = (2 * n_ + al + be) * (2 * n_ + al + be + 1) * (2 * n_ + al + be + 2)
            a4 = 2 * (n_ + al) * (n_ + be) * (2 * n_ + al + be + 2)
            a1 = a1 + 1e-4
            Pn1 = ((a2 + a3 * h) * basis[-1] - a4 * basis[-2]) / a1
            # No dynamic normalization - rely on LayerNorm
            basis.append(Pn1)
        R = torch.stack(basis, dim=-1)
        y = torch.einsum('bik,iok->bo', R, self.W)
        return y


class JacobiKAN(nn.Module):
    def __init__(self, dims, K, use_norm=True, dropout_rate=0.1):
        super().__init__()
        self.shape = JacobiShape()
        self.layers = nn.ModuleList()
        self.norms = nn.ModuleList()
        self.drops = nn.ModuleList()
        self.use_norm = use_norm
        
        for i in range(len(dims) - 1):
            self.layers.append(JacobiKANLayer(dims[i], dims[i + 1], K, self.shape))
            if i < len(dims) - 2:
                if use_norm:
                    self.norms.append(nn.LayerNorm(dims[i + 1]))
                self.drops.append(nn.Dropout(dropout_rate))
            else:
                if use_norm:
                    self.norms.append(nn.Identity())
                self.drops.append(nn.Identity())

    def forward(self, x):
        for i, layer in enumerate(self.layers):
            x = layer(x)
            if i < len(self.layers) - 1:
                x = self.norms[i](x)
                x = self.drops[i](x)
                x = torch.tanh(x)
        return x


def bspline_basis(x, grid, spline_order):
    # x: (B, d_in), grid: (d_in, G + 2*spline_order + 1)
    x = x.unsqueeze(-1)
    bases = ((x >= grid[:, :-1]) & (x < grid[:, 1:])).float()
    for k in range(1, spline_order + 1):
        left_num = x - grid[:, :-(k + 1)]
        left_den = grid[:, k:-1] - grid[:, :-(k + 1)]
        left = left_num / left_den.clamp_min(1e-6) * bases[:, :, :-1]
        right_num = grid[:, k + 1:] - x
        right_den = grid[:, k + 1:] - grid[:, 1:-k]
        right = right_num / right_den.clamp_min(1e-6) * bases[:, :, 1:]
        bases = left + right
    return bases  # (B, d_in, G+spline_order)


class SplineKANLayer(nn.Module):
    def __init__(self, d_in, d_out, grid_size, spline_order=3, grid_range=(-1.2, 1.2)):
        super().__init__()
        self.d_in, self.d_out = d_in, d_out
        self.grid_size, self.spline_order = grid_size, spline_order
        h = (grid_range[1] - grid_range[0]) / grid_size
        grid = torch.arange(-spline_order, grid_size + spline_order + 1) * h + grid_range[0]
        grid = grid.unsqueeze(0).repeat(d_in, 1)
        self.register_buffer('grid', grid)
        n_basis = grid_size + spline_order
        self.spline_W = nn.Parameter(torch.randn(d_in, d_out, n_basis) * (1.0 / math.sqrt(d_in * n_basis)))
        self.base_W = nn.Parameter(torch.randn(d_in, d_out) * (1.0 / math.sqrt(d_in)))
        self.scaler = nn.Parameter(torch.ones(d_in, d_out))

    def forward(self, x):
        h = x.clamp(-2.0, 2.0)  # Use clamp instead of tanh for fairness
        base = torch.nn.functional.silu(h)
        base_out = base @ self.base_W
        B_ = bspline_basis(h, self.grid, self.spline_order)
        spline_out = torch.einsum('bik,iok->bio', B_, self.spline_W)
        spline_out = (spline_out * self.scaler).sum(dim=1)
        return base_out + spline_out


class SplineKAN(nn.Module):
    def __init__(self, dims, grid_size, spline_order=3, use_norm=True, dropout_rate=0.1):
        super().__init__()
        self.layers = nn.ModuleList()
        self.norms = nn.ModuleList()
        self.drops = nn.ModuleList()
        self.use_norm = use_norm
        
        for i in range(len(dims) - 1):
            self.layers.append(SplineKANLayer(dims[i], dims[i + 1], grid_size, spline_order))
            if i < len(dims) - 2:
                if use_norm:
                    self.norms.append(nn.LayerNorm(dims[i + 1]))
                self.drops.append(nn.Dropout(dropout_rate))
            else:
                if use_norm:
                    self.norms.append(nn.Identity())
                self.drops.append(nn.Identity())

    def forward(self, x):
        for i, layer in enumerate(self.layers):
            x = layer(x)
            if i < len(self.layers) - 1:
                x = self.norms[i](x)
                x = self.drops[i](x)
                x = torch.tanh(x)
        return x


def count_params(model):
    return sum(p.numel() for p in model.parameters())


def match_spline_grid(dims, target_params, spline_orders=(1, 2, 3), search_range=range(1, 60)):
    """Search jointly over spline order and grid size for the (order, grid)
    pair whose total parameter count is closest to target_params.
    """
    best = None
    for order in spline_orders:
        for g in search_range:
            if g < 1:
                continue
            m = SplineKAN(dims, g, order, use_norm=False)  # Don't count norm params
            p = count_params(m)
            diff = abs(p - target_params)
            if best is None or diff < best[0]:
                best = (diff, order, g)
    return best[1], best[2]


DATA_DIR = os.environ.get('RECKAN_DATA_DIR', './data')
os.makedirs(DATA_DIR, exist_ok=True)


def mnist(seed=0):
    """Real, full MNIST via torchvision (downloads on first run)"""
    from torchvision import datasets, transforms
    tr = datasets.MNIST(DATA_DIR, train=True, download=True, transform=transforms.ToTensor())
    te = datasets.MNIST(DATA_DIR, train=False, download=True, transform=transforms.ToTensor())
    Xtr = tr.data.reshape(len(tr), -1).float() / 255.0
    ytr = tr.targets.long()
    Xte = te.data.reshape(len(te), -1).float() / 255.0
    yte = te.targets.long()
    return Xtr, ytr, Xte, yte, 'classification', 10


def cifar10(seed=0):
    """Real, full CIFAR-10 via torchvision"""
    from torchvision import datasets, transforms
    tr = datasets.CIFAR10(DATA_DIR, train=True, download=True, transform=transforms.ToTensor())
    te = datasets.CIFAR10(DATA_DIR, train=False, download=True, transform=transforms.ToTensor())
    Xtr = torch.tensor(tr.data, dtype=torch.float32).reshape(len(tr), -1) / 255.0
    ytr = torch.tensor(tr.targets, dtype=torch.long)
    Xte = torch.tensor(te.data, dtype=torch.float32).reshape(len(te), -1) / 255.0
    yte = torch.tensor(te.targets, dtype=torch.long)
    return Xtr, ytr, Xte, yte, 'classification', 10


def ag_news(seed=0, max_features=1000):
    """Real, full AG News text classification"""
    import pandas as pd
    from sklearn.feature_extraction.text import TfidfVectorizer

    train_url = "https://raw.githubusercontent.com/mhjabreel/CharCnn_Keras/master/data/ag_news_csv/train.csv"
    test_url = "https://raw.githubusercontent.com/mhjabreel/CharCnn_Keras/master/data/ag_news_csv/test.csv"
    cols = ['label', 'title', 'description']
    df_tr = pd.read_csv(train_url, header=None, names=cols)
    df_te = pd.read_csv(test_url, header=None, names=cols)

    text_tr = (df_tr['title'] + ' ' + df_tr['description']).tolist()
    text_te = (df_te['title'] + ' ' + df_te['description']).tolist()

    vec = TfidfVectorizer(max_features=max_features, stop_words='english')
    Xtr = vec.fit_transform(text_tr).toarray().astype(np.float32)
    Xte = vec.transform(text_te).toarray().astype(np.float32)
    ytr = (df_tr['label'].values - 1).astype(np.int64)
    yte = (df_te['label'].values - 1).astype(np.int64)

    return (torch.tensor(Xtr), torch.tensor(ytr), torch.tensor(Xte), torch.tensor(yte),
            'classification', 4)


def etth1(seed=0, lookback=96, horizon=1, test_frac=0.2):
    """Real ETTh1 (Electricity Transformer Temperature) forecasting"""
    import pandas as pd
    url = "https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTh1.csv"
    df = pd.read_csv(url)
    feats = df.drop(columns=['date']).values.astype(np.float32)
    target_col = feats.shape[1] - 1

    sx = StandardScaler().fit(feats)
    feats_n = sx.transform(feats).astype(np.float32)

    X, y = [], []
    for t in range(len(feats_n) - lookback - horizon + 1):
        X.append(feats_n[t:t + lookback].reshape(-1))
        y.append(feats_n[t + lookback:t + lookback + horizon, target_col])
    X = np.stack(X).astype(np.float32)
    y = np.stack(y).astype(np.float32)

    n_test = int(len(X) * test_frac)
    Xtr, Xte = X[:-n_test], X[-n_test:]
    ytr, yte = y[:-n_test], y[-n_test:]
    return (torch.tensor(Xtr), torch.tensor(ytr), torch.tensor(Xte), torch.tensor(yte),
            'regression', y.shape[1])


def ecg5000(seed=0):
    """Real ECG5000 heartbeat classification"""
    from aeon.datasets import load_classification
    Xtr, ytr_raw = load_classification("ECG5000", split="train")
    Xte, yte_raw = load_classification("ECG5000", split="test")
    Xtr = Xtr.reshape(Xtr.shape[0], -1).astype(np.float32)
    Xte = Xte.reshape(Xte.shape[0], -1).astype(np.float32)

    classes = sorted(np.unique(ytr_raw).tolist())
    remap = {c: i for i, c in enumerate(classes)}
    ytr = np.array([remap[c] for c in ytr_raw], dtype=np.int64)
    yte = np.array([remap[c] for c in yte_raw], dtype=np.int64)

    sx = StandardScaler().fit(Xtr)
    Xtr = sx.transform(Xtr).astype(np.float32)
    Xte = sx.transform(Xte).astype(np.float32)

    return (torch.tensor(Xtr), torch.tensor(ytr), torch.tensor(Xte), torch.tensor(yte),
            'classification', len(classes))


DATASETS = {
    'MNIST':   (mnist,   dict()),
    'CIFAR10': (cifar10, dict()),
    'AGNews':  (ag_news, dict(max_features=1000)),
    'ETTh1':   (etth1,   dict(lookback=96, horizon=1)),
    'ECG5000': (ecg5000, dict()),
}


# =====================================================================
# =====================================================================
#  PART 3 — TRAINING / COMPARISON
# =====================================================================
# =====================================================================

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
RESULTS_DIR = 'results'
os.makedirs(RESULTS_DIR, exist_ok=True)

# Improved architecture with deeper networks and better hyperparameters
ARCH = {
    'MNIST':   dict(hidden=30, K=3, epochs=50, batch_size=64, lr=1e-3, weight_decay=1e-4, dropout=0.1, use_norm=True),
    'CIFAR10': dict(hidden=64, K=3, epochs=50, batch_size=128, lr=1e-3, weight_decay=1e-4, dropout=0.1, use_norm=True),
    'AGNews':  dict(hidden=64, K=3, epochs=50, batch_size=128, lr=1e-3, weight_decay=1e-4, dropout=0.1, use_norm=True),
    'ETTh1':   dict(hidden=32, K=3, epochs=60, batch_size=64, lr=1e-3, weight_decay=1e-4, dropout=0.1, use_norm=True),
    'ECG5000': dict(hidden=32, K=3, epochs=60, batch_size=64, lr=1e-3, weight_decay=1e-4, dropout=0.1, use_norm=True),
}

# Update architectures with 3 layers (input -> hidden -> hidden -> output)
for name in ARCH:
    ARCH[name]['layers'] = 3  # Number of KAN layers


def build_model(name, dims, K, target_params, use_norm=True, dropout_rate=0.1):
    if name == 'SplineKAN':
        order, grid = match_spline_grid(dims, target_params)
        return SplineKAN(dims, grid, order, use_norm=use_norm, dropout_rate=dropout_rate)
    elif name == 'RecKAN':
        return RecKAN(dims, K, use_norm=use_norm, dropout_rate=dropout_rate)
    elif name == 'ChebyKAN':
        return ChebyKAN(dims, K, use_norm=use_norm, dropout_rate=dropout_rate)
    elif name == 'JacobiKAN':
        return JacobiKAN(dims, K, use_norm=use_norm, dropout_rate=dropout_rate)
    else:
        raise ValueError(f"Unknown model: {name}")


def iterate_batches(X, y, batch_size, shuffle=True):
    n = X.shape[0]
    idx = torch.randperm(n) if shuffle else torch.arange(n)
    for i in range(0, n, batch_size):
        b = idx[i:i + batch_size]
        yield X[b], y[b]


def train_one(model, Xtr, ytr, Xte, yte, task, epochs, batch_size, lr, weight_decay=1e-4):
    model.to(DEVICE)
    Xtr, ytr, Xte, yte = Xtr.to(DEVICE), ytr.to(DEVICE), Xte.to(DEVICE), yte.to(DEVICE)
    
    # Use AdamW with weight decay (improved from Adam)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    loss_fn = nn.CrossEntropyLoss() if task == 'classification' else nn.MSELoss()

    train_losses, test_metrics = [], []
    t0 = time.time()
    
    for ep in range(epochs):
        model.train()
        ep_loss, n_batches = 0.0, 0
        for xb, yb in iterate_batches(Xtr, ytr, batch_size):
            opt.zero_grad()
            out = model(xb)
            loss = loss_fn(out, yb)
            loss.backward()
            # Gentler gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            opt.step()
            ep_loss += loss.item()
            n_batches += 1
        train_losses.append(ep_loss / max(n_batches, 1))

        model.eval()
        with torch.no_grad():
            preds = []
            for xb, _ in iterate_batches(Xte, yte, batch_size, shuffle=False):
                preds.append(model(xb))
            preds = torch.cat(preds, dim=0)
            if task == 'classification':
                metric = (preds.argmax(dim=1) == yte).float().mean().item()
            else:
                metric = nn.functional.mse_loss(preds, yte).item()
        test_metrics.append(metric)
        
        scheduler.step()

    train_time = time.time() - t0
    return train_losses, test_metrics, train_time


def run_dataset(name, model_names, results_rows):
    print(f'\n=== {name} ===')
    loader_fn, kwargs = DATASETS[name]
    Xtr, ytr, Xte, yte, task, out_dim = loader_fn(**kwargs)
    in_dim = Xtr.shape[1]
    cfg = ARCH[name]
    
    # 3-layer architecture: input -> hidden -> hidden -> output
    hidden1 = cfg['hidden']
    hidden2 = cfg['hidden'] // 2  # Second hidden layer is smaller for efficiency
    dims = [in_dim, hidden1, hidden2, out_dim]
    
    print(f'  input_dim={in_dim}  output_dim={out_dim}  task={task}  '
          f'n_train={len(Xtr)}  n_test={len(Xte)}')

    # Reference model with the same architecture
    ref = RecKAN(dims, cfg['K'], use_norm=cfg['use_norm'], dropout_rate=cfg['dropout'])
    target_params = count_params(ref)
    print(f'  target parameter budget (RecKAN reference): {target_params}')

    fig, ax = plt.subplots(figsize=(6, 4))
    for mname in model_names:
        model = build_model(mname, dims, cfg['K'], target_params, 
                           use_norm=cfg['use_norm'], dropout_rate=cfg['dropout'])
        n_params = count_params(model)
        print(f'  training {mname:12s} params={n_params}')
        
        train_losses, test_metrics, train_time = train_one(
            model, Xtr, ytr, Xte, yte, task,
            epochs=cfg['epochs'], batch_size=cfg['batch_size'], 
            lr=cfg['lr'], weight_decay=cfg['weight_decay'])
            
        final_metric = test_metrics[-1]
        best_metric = max(test_metrics) if task == 'classification' else min(test_metrics)
        print(f'    -> final {"acc" if task=="classification" else "MSE"}='
              f'{final_metric:.4f}  best={best_metric:.4f}  time={train_time:.1f}s')

        ax.plot(train_losses, label=f'{mname} ({n_params}p)')
        results_rows.append(dict(
            dataset=name, model=mname, task=task, n_params=n_params,
            final_train_loss=train_losses[-1],
            final_test_metric=final_metric, best_test_metric=best_metric,
            train_time_sec=train_time))

    ax.set_yscale('log')
    ax.set_xlabel('epoch'); ax.set_ylabel('train loss (log scale)')
    ax.set_title(f'{name}: training loss')
    ax.legend()
    plt.tight_layout()
    plt.savefig(f'{RESULTS_DIR}/loss_curves_{name}.png', dpi=150)
    plt.close()


def make_summary_plot(results_rows, dataset_names, model_names):
    fig, axes = plt.subplots(1, len(dataset_names), figsize=(4 * len(dataset_names), 4))
    if len(dataset_names) == 1:
        axes = [axes]
    for ax, dname in zip(axes, dataset_names):
        rows = [r for r in results_rows if r['dataset'] == dname]
        task = rows[0]['task']
        vals = [next(r['final_test_metric'] for r in rows if r['model'] == m) for m in model_names]
        bars = ax.bar(model_names, vals, color=['#3b6fd6', '#999999', '#999999', '#999999'][:len(model_names)])
        for b, m in zip(bars, model_names):
            if m == 'RecKAN':
                b.set_color('#d64545')
        ax.set_title(dname)
        ax.set_ylabel('accuracy' if task == 'classification' else 'test MSE')
        ax.tick_params(axis='x', rotation=30)
    plt.tight_layout()
    plt.savefig(f'{RESULTS_DIR}/summary_bar.png', dpi=150)
    plt.close()


def write_csv(results_rows):
    path = f'{RESULTS_DIR}/results_table.csv'
    fields = ['dataset', 'model', 'task', 'n_params', 'final_train_loss',
              'final_test_metric', 'best_test_metric', 'train_time_sec']
    with open(path, 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=fields)
        w.writeheader()
        for r in results_rows:
            w.writerow(r)
    print(f'\nSaved {path}')


def print_summary_table(results_rows):
    print('\n' + '=' * 78)
    print(f"{'dataset':10s} {'model':11s} {'params':>8s} {'metric':>10s} {'best':>10s} {'time(s)':>8s}")
    print('-' * 78)
    for r in results_rows:
        print(f"{r['dataset']:10s} {r['model']:11s} {r['n_params']:8d} "
              f"{r['final_test_metric']:10.4f} {r['best_test_metric']:10.4f} "
              f"{r['train_time_sec']:8.1f}")
    print('=' * 78)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--datasets', nargs='+', default=list(DATASETS.keys()))
    parser.add_argument('--models', nargs='+',
                         default=['RecKAN', 'ChebyKAN', 'JacobiKAN', 'SplineKAN'])
    args, unknown = parser.parse_known_args()

    torch.manual_seed(0)
    np.random.seed(0)

    results_rows = []
    for dname in args.datasets:
        run_dataset(dname, args.models, results_rows)

    write_csv(results_rows)
    print_summary_table(results_rows)
    make_summary_plot(results_rows, args.datasets, args.models)
    print(f'\nAll plots saved under ./{RESULTS_DIR}/')


if __name__ == '__main__':
    main()
